# OS-A02-A06 — Executable Classification Laboratory
## AI Concepts for Official Statisticians

**Controlled learner asset v1.0**  
This laboratory turns an official-statistics task description into an evidence-bounded recommendation about an AI mechanism. It does **not** authorize production use. All records are synthetic and contain no personal or confidential information.

## Learning outcomes

By completing the laboratory, the learner can: (1) decompose a statistical task; (2) distinguish rules, supervised learning, generative/retrieval systems, optimization and a justified no-AI option; (3) compare a transparent baseline with a learned classifier; (4) test population, subgroup and shifted-data performance; (5) trace failure propagation; and (6) issue a bounded decision with recorded conditions.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
warnings.filterwarnings('ignore')
SEED=20260926
np.random.seed(SEED)
BASE=Path.cwd()
if not (BASE/'OS-A02-A06_Kintaba_Task_Classification_Data_v1.0.csv').exists():
    BASE=Path('/workspace/scratch/01b4d521a2db/deliverables/OS-A02-A06-v1.0')
DATA=BASE/'OS-A02-A06_Kintaba_Task_Classification_Data_v1.0.csv'
STRESS=BASE/'OS-A02-A06_Kintaba_Stress_Test_Data_v1.0.csv'
print({'python_seed':SEED,'data_found':DATA.exists(),'stress_found':STRESS.exists()})

## 1. Evidence contract

The target is a **mechanism class**, not a claim that AI should be deployed. The label encodes a defensible starting hypothesis from the accepted lesson. A deployment decision additionally requires legal authority, confidentiality controls, statistical-quality testing, reproducibility, human review and operational monitoring.

In [ ]:
df=pd.read_csv(DATA)
assert df['task_id'].is_unique and len(df)==360
assert set(df.true_mechanism)=={'rule_based','supervised_learning','generative_retrieval','optimization','no_ai'}
print(df.shape)
display(df.head(8))

## 2. Data dictionary and task decomposition

`production_stage` locates the task in the statistical process. `input_type` and `objective` describe the technical operation. `repeatability` approximates whether a stable pattern exists. `uncertainty_level`, `decision_authority` and `sensitivity` describe consequences and governance constraints. `region` is retained only to test distributional performance; it is not a causal justification for mechanism choice.

In [ ]:
summary=pd.DataFrame({
 'dtype':df.dtypes.astype(str),
 'missing':df.isna().sum(),
 'unique':df.nunique()
})
display(summary)
print(df.true_mechanism.value_counts())

## 3. Independent split

We hold back 30% of records using stratification. The test set must not influence fitting. This is necessary but not sufficient: the later stress set represents a distinct operational challenge.

In [ ]:
features=['region','production_stage','input_type','objective','repeatability','uncertainty_level','decision_authority','sensitivity']
X=df[features]; y=df.true_mechanism
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.30,random_state=SEED,stratify=y)
print({'train':len(X_train),'test':len(X_test),'train_share':round(len(X_train)/len(df),3)})

## 4. Transparent rule baseline

The baseline represents explicit institutional reasoning. It is inspectable and can reveal whether a learned model adds value. Its precedence matters: high-uncertainty release decisions are routed to `no_ai` before convenience or automation is considered.

In [ ]:
def rule_baseline(r):
    if r.decision_authority=='statistical_release' and r.uncertainty_level=='high': return 'no_ai'
    if r.objective=='allocate': return 'optimization'
    if r.input_type=='text' and r.objective in ['draft','retrieve']: return 'generative_retrieval'
    if r.objective in ['classify','predict'] and r.repeatability>=3: return 'supervised_learning'
    if r.repeatability>=4 and r.uncertainty_level=='low': return 'rule_based'
    return 'no_ai'
base_pred=X_test.apply(rule_baseline,axis=1)
base_f1=f1_score(y_test,base_pred,average='macro')
print({'baseline_macro_f1':round(base_f1,3)})
display(pd.DataFrame(confusion_matrix(y_test,base_pred,labels=sorted(y.unique())),index=sorted(y.unique()),columns=sorted(y.unique())))

## 5. Learned classifier

The model uses one-hot encoding for categorical fields and standardization for repeatability. Multinomial logistic regression is intentionally modest: it is reproducible, supports class probabilities and permits comparison with the explicit rule system.

In [ ]:
categorical=[c for c in features if c!='repeatability']
prep=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),categorical),('num',StandardScaler(),['repeatability'])])
model=Pipeline([('prep',prep),('clf',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))])
model.fit(X_train,y_train)
pred=model.predict(X_test); prob=model.predict_proba(X_test)
model_f1=f1_score(y_test,pred,average='macro')
print({'model_macro_f1':round(model_f1,3),'increment_over_baseline':round(model_f1-base_f1,3)})
print(classification_report(y_test,pred,digits=3))

## 6. Confusion structure

An aggregate score can conceal a consequential class failure. The confusion matrix is read by true class (rows) and predicted class (columns). Errors involving `no_ai` deserve particular scrutiny because they can turn a governance restriction into an automation recommendation.

In [ ]:
classes=list(model.classes_)
cm=confusion_matrix(y_test,pred,labels=classes)
fig,ax=plt.subplots(figsize=(8,5)); im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=range(len(classes)),yticks=range(len(classes)),xticklabels=classes,yticklabels=classes,xlabel='Predicted',ylabel='True',title='Held-out confusion matrix')
plt.setp(ax.get_xticklabels(),rotation=30,ha='right')
for i in range(len(classes)):
    for j in range(len(classes)): ax.text(j,i,cm[i,j],ha='center',va='center')
plt.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## 7. Confidence and abstention

Probability is not institutional confidence. Here it supports an abstention rule: cases below the threshold are referred for expert review. Coverage must be reported with performance so that selective accuracy is not mistaken for system-wide quality.

In [ ]:
maxp=prob.max(axis=1); threshold=.72; accepted=maxp>=threshold
selective_f1=f1_score(y_test[accepted],pred[accepted],average='macro') if accepted.any() else np.nan
print({'threshold':threshold,'coverage':round(accepted.mean(),3),'abstention_rate':round(1-accepted.mean(),3),'selective_macro_f1':round(selective_f1,3)})

## 8. Subgroup audit

We evaluate regions separately even though region is synthetic. The aim is methodological: a national average cannot demonstrate fitness for every production context. Small subgroup counts must be interpreted cautiously and reported rather than silently pooled.

In [ ]:
audit=X_test[['region']].copy(); audit['truth']=y_test.values; audit['prediction']=pred
group=[]
for g,z in audit.groupby('region'):
    group.append({'region':g,'n':len(z),'macro_f1':f1_score(z.truth,z.prediction,average='macro')})
group=pd.DataFrame(group).sort_values('region'); disparity=group.macro_f1.max()-group.macro_f1.min()
display(group); print({'regional_f1_range':round(disparity,3)})

## 9. Consequential-class recall

For this laboratory, `no_ai` recall is a safety-oriented diagnostic. Low recall means the classifier often fails to preserve a deliberate non-automation decision. It is not the only quality criterion, but it cannot be averaged away.

In [ ]:
recalls=recall_score(y_test,pred,labels=classes,average=None)
recall_table=pd.DataFrame({'class':classes,'recall':recalls})
display(recall_table)
print({'minimum_class_recall':round(recall_table.recall.min(),3),'no_ai_recall':round(float(recall_table.loc[recall_table['class']=='no_ai','recall'].iloc[0]),3)})

## 10. Operational stress test

The stress set introduces an unseen region and shifts some input and uncertainty values. This is not a forecast of real deployment. It asks whether the pipeline handles new categories and whether performance deteriorates under a plausible boundary change.

In [ ]:
stress=pd.read_csv(STRESS); Xs=stress[features]; ys=stress.true_mechanism
sp=model.predict(Xs); stress_f1=f1_score(ys,sp,average='macro')
print({'stress_n':len(stress),'stress_macro_f1':round(stress_f1,3),'change_from_test':round(stress_f1-model_f1,3)})
display(pd.DataFrame(confusion_matrix(ys,sp,labels=classes),index=classes,columns=classes))

## 11. Failure-propagation register

A wrong mechanism recommendation can propagate through procurement, system design, evaluation, production and dissemination. The register links each observed failure mode to a control owned by a named institutional function.

In [ ]:
failure_register=pd.DataFrame([
 ['Misclassify no-AI case','Unsafe automation candidate','Methodologist review plus hard authority rule','Methodology'],
 ['Unknown operational category','Unstable or arbitrary mapping','Schema validation and abstention','Data engineering'],
 ['Regional performance gap','Unequal error burden','Stratified evaluation and remediation','Quality management'],
 ['Low probability margin','False precision','Referral threshold and recorded disposition','Production owner'],
 ['Stress degradation','Boundary failure after rollout','Shadow operation and drift monitoring','System owner']
],columns=['failure_mode','downstream_effect','control','owner'])
display(failure_register)

## 12. Release gate

The gate combines independent criteria. Passing it only supports a controlled pilot recommendation. It does not establish legality, statistical validity or public-release authority. Those remain human institutional responsibilities.

In [ ]:
criteria={
 'heldout_macro_f1_at_least_0_72':bool(model_f1>=.72),
 'minimum_class_recall_at_least_0_60':bool(recall_table.recall.min()>=.60),
 'regional_f1_range_at_most_0_15':bool(disparity<=.15),
 'stress_macro_f1_at_least_0_65':bool(stress_f1>=.65),
 'no_ai_recall_at_least_0_80':bool(float(recall_table.loc[recall_table['class']=='no_ai','recall'].iloc[0])>=.80),
}
decision='CONTROLLED PILOT CANDIDATE' if all(criteria.values()) else 'REVISE BEFORE PILOT'
print(json.dumps(criteria,indent=2)); print('DECISION:',decision)

## 13. Required human determination

Before any institutional use, the subject-matter statistician must confirm the target definition, population, loss consequences, tolerances and link to the statistical quality framework. Legal and security specialists must confirm authority, confidentiality and access controls. The production owner must approve monitoring, incident response, rollback and versioning. The model may structure evidence; it cannot assume these accountabilities.

In [ ]:
result={
 'asset':'OS-A02-A06','version':'1.0','records':len(df),'heldout_records':len(X_test),
 'baseline_macro_f1':round(base_f1,4),'model_macro_f1':round(model_f1,4),
 'stress_macro_f1':round(stress_f1,4),'regional_f1_range':round(disparity,4),
 'decision':decision,'public_release_authorized':False
}
RESULT=BASE/'OS-A02-A06_reference_results.json'
RESULT.write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps(result,indent=2))

## Reflection and transfer

1. Which fields describe the statistical task, and which fields describe institutional consequences?  
2. When does the rule baseline remain preferable even if the learned model scores higher?  
3. What tolerance would your institution set for a false automation recommendation?  
4. Which real workplace task could be represented without using confidential records?  
5. What evidence would be required to move from a laboratory result to a controlled pilot?

**Source basis:** SRAI OS-A02-A01 v0.2 and accepted OS-A02-A02–A05 derived assets. Synthetic data are pedagogical and must not be interpreted as official statistics.